# SMM Estimation
Structural estimation of `FullLaborModelClass` via Simulated Method of Moments.

**Workflow**
1. Build data moments from register data
2. Verify moment function at current parameters
3. Stage 1 — Differential Evolution (global search)
4. Stage 2 — Nelder-Mead polish
5. Optimal weighting matrix (bootstrap)
6. Re-estimate with optimal W
7. Standard errors and moment fit table


## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from DynLaborSickModel import FullLaborModelClass
from Estimation import (
    SMMEstimator,
    make_data_moments,
    moment_fit_table,
    PARAM_SPEC,
)


## Initialise estimator

In [ ]:
est = SMMEstimator(
    FullLaborModelClass,
    calibrated = {
        'beta':  0.98,
        'tau':   0.30,
        'b_wel': 0.40,
        'Ubar':  24,
    },
)

# Show parameter specification
pd.DataFrame(PARAM_SPEC, columns=['name','lower','upper','description'])


## Data moments

Load your empirical hazard rates and scalar moments here.
Each hazard DataFrame needs columns `duration` (int) and `hazard` (float).
Leave any unavailable moment as `None` — it will be excluded from the objective.

In [ ]:
# ── Load empirical moments (replace with actual data paths) ───────────────
# hz_ue_data      = pd.read_csv('moments/hazard_ue.csv')       # duration, hazard
# hz_us_data      = pd.read_csv('moments/hazard_us.csv')
# hz_se_data      = pd.read_csv('moments/hazard_se.csv')
# hz_su_data      = pd.read_csv('moments/hazard_su.csv')
# hz_se_Eorig_data= pd.read_csv('moments/hazard_se_Eorig.csv')
# hz_su_Uorig_data= pd.read_csv('moments/hazard_su_Uorig.csv')

data_moments = make_data_moments(
    hz_ue_df         = None,   # replace with hz_ue_data
    hz_us_df         = None,
    hz_se_df         = None,
    hz_su_df         = None,
    hz_se_Eorig_df   = None,
    hz_su_Uorig_df   = None,
    avg_u_dur        = None,   # scalar: mean unemployment duration
    avg_s_dur        = None,   # scalar: mean sick-leave duration
    share_E          = None,   # scalar: employment share
    share_U          = None,   # scalar: unemployment share
    share_S          = None,   # scalar: sick-leave share
    share_S_Eorig    = None,   # scalar: share of sick leave from employment
    exhaustion_spike = None,   # scalar: U→E hazard ratio at exhaustion
)
print(f'{len(data_moments)} data moments loaded')


## Verify moment function at current parameters

Solves and simulates the model once at the default parameter values
to confirm all moments compute correctly before running estimation.

In [ ]:
# Build theta0 from default model parameters
_m = FullLaborModelClass()
_m.setup()
_p = _m.par

theta0 = [
    _p.psi,
    _p.gamma,
    _p.iota,
    _p.chi,
    _p.alpha,
    _p.delta_k,
    _p.rho_h,
    _p.sigma_h,
    _p.delta_h_S,
    _p.delta0_doc,
    _p.delta1_doc,
    _p.b_sick_low,
    _p.lambda_grid[0],
    _p.lambda_grid[1],
    _p.nu_grid[0],
    _p.nu_grid[1],
    _p.type_shares[1],
    _p.h_init_mu[1],
]

m0 = est.compute_moments(theta0)
pd.Series(m0, name='value').sort_index().to_frame()


## Stage 1 + 2: Estimate (run when data moments are ready)

In [ ]:
# result = est.estimate(
#     data_moments,
#     de_maxiter         = 500,
#     de_popsize         = 15,
#     de_seed            = 42,
#     polish_with_nelder = True,
#     verbose            = False,
# )
# print(result['table'])


## Optimal weighting matrix + second-stage re-estimation

In [ ]:
# W_opt = est.optimal_weight_matrix(
#     result['theta'], data_moments, n_bootstrap=200
# )

# result2 = est.estimate(
#     data_moments,
#     W          = W_opt,
#     de_maxiter = 300,
# )


## Standard errors and moment fit

In [ ]:
# table = est.results_table(
#     result2['theta'],
#     data_moments = data_moments,
#     W            = W_opt,
#     compute_se   = True,
# )

# fit = moment_fit_table(
#     est.compute_moments(result2['theta']),
#     data_moments,
# )
# display(fit)
